# Actual NPV Analysis — Submission A

Computes realized NPV for every labeled val row (where `default_flag` is known).

**NPV formulas** (same as deliverable_A.ipynb Step 12):
```
fee            = 0.03 × requested_amount
interest       = requested_amount × 0.35 × 60 / 365
daily_draw     = requested_amount × (1 + 0.35×60/365) / 60

NPV_repaid  = fee + interest
NPV_default = fee + daily_draw × (default_day − 1) + recovered_amount − requested_amount
```

**Realized NPV per row:**
- `decision = 0` (declined)  → NPV = 0  (no loan made)
- `decision = 1, default_flag = 0` (approved, repaid) → NPV_repaid
- `decision = 1, default_flag = 1` (approved, defaulted) → NPV_default

Only rows where `default_flag` is not null are included (2,551 of 4,489 val rows).

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)

## 1. Load data

In [17]:
val = pd.read_csv('dataset/dataset-compressed/validation.csv')
val = val[~val.default_flag.isna()].reset_index(drop=True)

In [19]:
val = val[['applicant_id','requested_amount','days_to_default','final_recovered_amount','default_flag']]

In [20]:
val

,applicant_id,requested_amount,days_to_default,final_recovered_amount,default_flag
0,cd890057-48c7-c041-56cf-190f86f6d8d3,"22,299.97",NaN,NaN,0.00
1,b802b6d4-e4b0-b4e6-18ac-7e308e85085b,"31,212.66",60.00,0.00,1.00
2,aa35bcc5-1f0d-b0ab-d18a-1e407df8608a,"24,810.91",NaN,NaN,0.00
3,bed7a0f6-843c-5d12-51dd-f83c75557255,"22,322.08",NaN,NaN,0.00
4,0a73b6a5-9886-207e-5495-6d080904b531,"26,535.83",NaN,NaN,0.00
...,...,...,...,...,...
2546,fac64b38-4d39-5a98-1e5b-1dd98af1f148,"33,265.41",NaN,NaN,0.00
2547,68c9b368-abd2-ee8b-9a49-44d9013d091b,"26,128.01",NaN,NaN,0.00
2548,243ae854-661a-14e9-d584-6e28f2729481,"18,740.58",NaN,NaN,0.00
2549,1ccbc7d7-a0ca-ec03-02c9-769182c5136e,"35,395.33",14.00,"13,299.49",1.00


In [21]:
def calculate_npv_repaid(requested_amt, fee_ratio=0.03, r=0.35, T=60):
    F = fee_ratio * requested_amt
    return F + (requested_amt * r * T / 365)

def calculate_npv_default(requested_amt, default_day, recovered_amt,
                           fee_ratio=0.03, r=0.35, T=60, min_missed_draws=3):
    """
    NPV when a loan defaults on `default_day`.

    default_day is the day default was declared (trigger: 3 consecutive missed
    draws, 6 total missed draws, or balance > 0 at day 90). At minimum,
    min_missed_draws (=3) ACH draws were not collected before declaration.

    successful_draws = max(0, default_day - min_missed_draws)
    """
    F              = fee_ratio * requested_amt
    daily_draw     = requested_amt * (1 + (r * T / 365)) / T
    successful     = np.maximum(0, default_day - min_missed_draws)
    return F + daily_draw * successful + recovered_amt - requested_amt


def app(x):
    if x['default_flag'] == 1:
        return calculate_npv_default(x['requested_amount'],x['days_to_default'],x['final_recovered_amount'])
    else:
        return calculate_npv_repaid(x['requested_amount'])

In [24]:
val['npv_calculated'] = val.apply(lambda row: app(row),axis=1)
val

/var/folders/ct/vwsnn9wd3wjbn9chxs6nswy80000gn/T/ipykernel_17887/1998569423.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val['npv_calculated'] = val.apply(lambda row: app(row),axis=1)


,applicant_id,requested_amount,days_to_default,final_recovered_amount,default_flag,npv_calculated
0,cd890057-48c7-c041-56cf-190f86f6d8d3,"22,299.97",NaN,NaN,0.00,"1,952.01"
1,b802b6d4-e4b0-b4e6-18ac-7e308e85085b,"31,212.66",60.00,0.00,1.00,"1,081.75"
2,aa35bcc5-1f0d-b0ab-d18a-1e407df8608a,"24,810.91",NaN,NaN,0.00,"2,171.80"
3,bed7a0f6-843c-5d12-51dd-f83c75557255,"22,322.08",NaN,NaN,0.00,"1,953.95"
4,0a73b6a5-9886-207e-5495-6d080904b531,"26,535.83",NaN,NaN,0.00,"2,322.79"
...,...,...,...,...,...,...
2546,fac64b38-4d39-5a98-1e5b-1dd98af1f148,"33,265.41",NaN,NaN,0.00,"2,911.86"
2547,68c9b368-abd2-ee8b-9a49-44d9013d091b,"26,128.01",NaN,NaN,0.00,"2,287.10"
2548,243ae854-661a-14e9-d584-6e28f2729481,"18,740.58",NaN,NaN,0.00,"1,640.44"
2549,1ccbc7d7-a0ca-ec03-02c9-769182c5136e,"35,395.33",14.00,"13,299.49",1.00,"-14,171.49"


In [26]:
val[val.npv_calculated>0].npv_calculated.sum()

np.float64(6407489.409458261)

In [27]:
sub = pd.read_csv('submissions/submission_A.csv')

In [31]:
sub = sub[sub.decision==1].reset_index()

In [32]:
merged = sub.merge(val, on='applicant_id', how='inner')
merged

,index,applicant_id,decision,predicted_pd,pd_lower_90,pd_upper_90,requested_amount,days_to_default,final_recovered_amount,default_flag,npv_calculated
0,4,cd890057-48c7-c041-56cf-190f86f6d8d3,1,0.11,0.11,0.12,"22,299.97",NaN,NaN,0.00,"1,952.01"
1,9,aa35bcc5-1f0d-b0ab-d18a-1e407df8608a,1,0.15,0.15,0.15,"24,810.91",NaN,NaN,0.00,"2,171.80"
2,10,bed7a0f6-843c-5d12-51dd-f83c75557255,1,0.11,0.11,0.11,"22,322.08",NaN,NaN,0.00,"1,953.95"
3,11,0a73b6a5-9886-207e-5495-6d080904b531,1,0.15,0.13,0.17,"26,535.83",NaN,NaN,0.00,"2,322.79"
4,28,1c74a6ee-51c0-956a-2823-34618eb9e597,1,0.28,0.27,0.30,"33,351.07",NaN,NaN,0.00,"2,919.36"
...,...,...,...,...,...,...,...,...,...,...,...
1962,4479,f00f0f3b-5303-8b3d-1948-c546d5f44e23,1,0.07,0.05,0.10,"21,265.21",NaN,NaN,0.00,"1,861.43"
1963,4480,322abd32-28c9-2dd9-76b3-28fa4614760b,1,0.10,0.08,0.12,"27,722.59",NaN,NaN,0.00,"2,426.68"
1964,4483,68c9b368-abd2-ee8b-9a49-44d9013d091b,1,0.11,0.08,0.14,"26,128.01",NaN,NaN,0.00,"2,287.10"
1965,4486,243ae854-661a-14e9-d584-6e28f2729481,1,0.04,0.02,0.07,"18,740.58",NaN,NaN,0.00,"1,640.44"


In [33]:
merged.npv_calculated.sum()

np.float64(3598587.0180577682)